In [1]:
import pandas as pd
import numpy as np
import netwulf as nw
import networkx as nx
import re
import string

import nltk
from nltk.corpus import stopwords

from itertools import combinations
from collections import Counter

nltk.download('stopwords', quiet=True)   # must happen before first use
stop_words = set(stopwords.words('english'))

In [5]:
df = pd.read_csv("../scraped_data/scraped_data.csv")
df = df[~df.title.isna()]
df["text"][0:2]

0    Trump to give a non-State of the Union address...
1    TORONTO, March 04, 2025 (GLOBE NEWSWIRE) -- In...
Name: text, dtype: object

In [58]:
stop_words.update(["news","breaking","latest","today","video","headlines","world","would","says","near","towards","around"])
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'around',
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'breaking',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'headlines',
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'latest',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'near',
 'needn',
 "needn't",
 'news',
 'no'

In [60]:
phrases = [
    "hong kong",
    "xi jinping",
    "artificial intelligence",
    "united states",
    "south china sea",
    "taiwan strait",
    "new zealand",
    "new york",
    "kamala harris",
    "supply chain",
    "social media",
    "air force",
    "south korea",
    "north korea",
    "nancy pelosi",
    "joe biden",
    "donald trump",
    "white house",
    "jets fighter",
    "middle east",
    "suez canal",
    "dwight howard",
    "tokyo olympics",
    "supreme court",
]

In [62]:
def merge_phrases(text, phrases):
    for phrase in phrases:
        text = text.replace(phrase, phrase.replace(" ", "_"))
    return text

In [64]:
df["clean_title"] = df["title"].str.lower().apply(lambda x: merge_phrases(x, phrases))


In [65]:
# df["split_title"] = df.title.apply(lambda x: x.split(" "))
punct = string.punctuation.replace("_", "")
df["split_title"] = df["clean_title"].apply(
    lambda x: [
        word for word in re.sub(f"[{re.escape(punct)}]", "", x.lower()).split()
        if word not in stop_words
    ]
)


In [66]:
[word for word in re.sub(f"[{re.escape(punct)}]", "", df.title.iloc[0].lower()).split() if word not in stop_words]

['douglas',
 'andrews',
 'thomas',
 'gallatin',
 'jordan',
 'candler',
 'tuesday',
 'fold']

In [67]:
df["word_pairs"] = df.split_title.apply(lambda x: list(combinations(sorted(set(x)),2)))

In [68]:
flat_list = [x for xs in df.word_pairs for x in xs]

In [69]:
EL = Counter(flat_list) 

In [70]:
EL2 = EL.most_common(1000)
EL2 = [(e[0][0], e[0][1], e[1]) for e in EL2]
EL2 = pd.DataFrame(EL2, columns = ["u1","u2","w"])


for col in ["u1", "u2"]:
    EL2[col] = EL2[col].astype(str).str.lower()

In [ ]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

for col in ["u1", "u2"]:
    EL2[col] = EL2[col].astype(str).apply(
        lambda x: ' '.join([word for word in x.split() if word not in stop_words]))
print(EL2)

In [72]:
G = nx.Graph()
G.add_weighted_edges_from(EL2.values)

In [77]:
import community as community_louvain
partition = community_louvain.best_partition(G, weight='weight')
nx.set_node_attributes(G, partition, 'community')


In [ ]:
nw.visualize(G)

In [ ]:
import nltk
import networkx as nx
from nltk.corpus import stopwords
from collections import Counter
import community as community_louvain
from netwulf import visualize
import pandas as pd

# Setup
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# 1. Remove stopwords carefully from u1 and u2 (leave merged phrases intact)
def remove_stopwords(text):
    if pd.isna(text): return ""
    return "_".join([w for w in text.lower().split("_") if w not in stop_words])

for col in ["u1", "u2"]:
    EL2[col] = EL2[col].astype(str).apply(remove_stopwords)

# 2. Remove rows where a node is now empty
EL2 = EL2[(EL2["u1"] != "") & (EL2["u2"] != "")]

# 3. Build the network
G = nx.Graph()
edges = [tuple(x) for x in EL2[["u1", "u2", "w"]].values]
G.add_weighted_edges_from(edges)

# 4. Detect communities
partition = community_louvain.best_partition(G, weight='weight')
nx.set_node_attributes(G, partition, 'community')

# 5. Visualize with Netwulf
visualize(G)
